**Выполнил: Александр Рябков, P3123**

# Лабораторная работа 8. Скрапинг

##  Цель работы
Освоение методов скрапинга данных из веб-страниц.

## Теоретические сведения
Основные понятия:

Скрейпинг / Веб-скрейпинг — это технология получения веб-данных путём извлечения их со страниц веб-ресурсов.


Объект для скрапинга: https://news.itmo.ru/ru.

Необходимо сохранить полученные данных в формате csv внутри колаба.

Структура данных :

1. Идентификатор новости (целочисленное число из URL)
2. Название новости
3. Дата её размещения
3. URL на страницу с конкретной новостью.

In [71]:
import csv
import requests
from bs4 import BeautifulSoup


In [72]:
DOMAIN = 'https://news.itmo.ru'
SEARCH_DOMAIN = 'https://news.itmo.ru/ru/search/?search='

In [ ]:
headers = []
urls = []

queries = ['нейротехнологии', 'нейротехнологии и программирование']

"""
<li class="weeklyevent"><h4><a href="/ru/education/cooperation/news/14792/">Более 400 участников, 30 регионов и 6 треков: главные итоги и цифры рекордного сезона финалов НТО в ИТМО</a></h4><p>… Native»
	«Компьютерные системы и технологии»
	«Разработка программного обеспечения»
	«Инженерия искусственного интеллекта»
	«Языковые модели и искусственный интеллект»
	«<span class="selection">Нейротехнологии и программирование</span>»
	«Системное и прикладное программное обеспечение»
|s=2,center|
«Летающая робототехника» ― 49 финалистов. Партнеры ― «Клевер COEX», «Свеза» и Acadе…</p><p>06.04.2026</p></li>
"""

for query in queries:
    data = requests.get(SEARCH_DOMAIN + query)
    bs = BeautifulSoup(data.text, 'html.parser')

    # общее кол-во найденных новостей
    total_news = bs.find('div', {'class': 'weeklyevents'})
    total_news_count = int(total_news.find('h2').find('span').get_text().split()[0])

    # берем заголовок и ссылку каждой новости
    news_headers = bs.find_all('li', {'class': 'weeklyevent'})
    for _n in news_headers:
        post = _n.find('h4').find('a').get_text()
        headers.append(post)
        post_url = _n.find('h4').find('a').get('href')
        urls.append(DOMAIN + post_url)

print(len(urls))

In [74]:
urls
# 20 записей (часть из них дублируется, т r оба запроса находят одни и те же новости)

['https://news.itmo.ru/ru/education/cooperation/news/14792/',
 'https://news.itmo.ru/ru/education/trend/news/14771/',
 'https://news.itmo.ru/ru/education/official/news/14666/',
 'https://news.itmo.ru/ru/education/trend/news/14508/',
 'https://news.itmo.ru/ru/education/official/news/14481/',
 'https://news.itmo.ru/ru/university_live/ratings/news/14438/',
 'https://news.itmo.ru/ru/education/trend/news/14434/',
 'https://news.itmo.ru/ru/education/cooperation/news/14405/',
 'https://news.itmo.ru/ru/university_live/achievements/news/14290/',
 'https://news.itmo.ru/ru/education/trend/news/13699/',
 'https://news.itmo.ru/ru/education/cooperation/news/14792/',
 'https://news.itmo.ru/ru/education/trend/news/14771/',
 'https://news.itmo.ru/ru/education/official/news/14666/',
 'https://news.itmo.ru/ru/education/trend/news/14508/',
 'https://news.itmo.ru/ru/education/official/news/14481/',
 'https://news.itmo.ru/ru/university_live/ratings/news/14438/',
 'https://news.itmo.ru/ru/education/trend/new

In [75]:
print(headers[:2])
print(urls[:2])

['Более 400 участников, 30 регионов и 6 треков: главные итоги и цифры рекордного сезона финалов НТО в ИТМО', 'Топ образование по программированию и ИИ в 2026-м: на что смотреть и как выбирать']
['https://news.itmo.ru/ru/education/cooperation/news/14792/', 'https://news.itmo.ru/ru/education/trend/news/14771/']


In [ ]:
import csv
import requests
from bs4 import BeautifulSoup
import random
import time
import re

news_data = []
output_csv_file = 'scraped_news_data.csv'

print("Начинается сбор даты для каждой новости. Это может занять некоторое время...")

for i, (header, url) in enumerate(zip(headers, urls)):
    news_id = 'N/A'
    news_date = 'Дата не найдена'

    # ID из URL
    match = re.search(r'/(\d+)/?$', url)
    if match:
        news_id = match.group(1)

    try:
        time.sleep(random.randint(0, 1))
        response = requests.get(url)
        response.raise_for_status()
        soup_news = BeautifulSoup(response.text, 'html.parser')

        # дата в <div class="time"><time>
        date_element = soup_news.find('div', class_='time')
        if date_element and date_element.find('time'):
            news_date = date_element.find('time').get_text(strip=True)

    except requests.exceptions.RequestException as e:
        print(f"Ошибка при запросе {url}: {e}")
    except Exception as e:
        print(f"Ошибка при парсинге {url}: {e}")

    news_data.append({
        'Идентификатор новости': news_id,
        'Название новости': header,
        'Дата её размещения': news_date,
        'URL на страницу с конкретной новостью': url
    })

    if (i + 1) % 10 == 0:
        print(f"Обработано {i + 1} новостей...")

print("Сбор данных завершен. Начинается запись в CSV.")

fieldnames = [
    'Идентификатор новости',
    'Название новости',
    'Дата её размещения',
    'URL на страницу с конкретной новостью'
]

try:
    with open(output_csv_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(news_data)
    print(f"Данные успешно записаны в файл {output_csv_file}")
except IOError as e:
    print(f"Ошибка при записи в файл {output_csv_file}: {e}")

In [ ]:
# считаем кол-во страниц в /ru/main_news/
# когда кнопка "вперед" становится "#" — значит последняя страница

URL = 'https://news.itmo.ru/ru/main_news/'
total_pages = None

for i in range(1, 200):
    for attempt in range(3):
        try:
            data = requests.get(URL + str(i) + '/', timeout=15)
            break
        except Exception as e:
            print(f'Страница {i}, попытка {attempt+1} не удалась: {e}')
            time.sleep(3)
    else:
        print(f'Страница {i} недоступна, пропускаем')
        continue

    print(data.url)

    page = BeautifulSoup(data.text, 'html.parser')

    next_page_href = page.find_all('div', {'class': 'pagination'})[0]\
                                  .find('ul')\
                                  .find_all('li')[1]\
                                  .find('a')['href']

    if next_page_href == '#':
        total_pages = i
        break

    time.sleep(random.randint(0, 1))

print(total_pages)

In [ ]:
# собираем список новостей с /ru/main_news/
# структура: ul.triplet > li > h4 > a (заголовок + ссылка), time (дата)

import re
import time
import requests
import csv
from bs4 import BeautifulSoup

DOMAIN = 'https://news.itmo.ru'
MAIN_NEWS_URL = 'https://news.itmo.ru/ru/main_news/'
PAGES = 5  # кол-во страниц (по ~9 новостей каждая)

news_list = []

for page_num in range(1, PAGES + 1):
    # до 3 попыток на случай таймаута
    for attempt in range(3):
        try:
            response = requests.get(MAIN_NEWS_URL + str(page_num) + '/', timeout=15)
            response.encoding = 'utf-8'
            break
        except Exception as e:
            print(f'Страница {page_num}, попытка {attempt+1} не удалась: {e}')
            time.sleep(3)
    else:
        print(f'Страница {page_num} недоступна, пропускаем')
        continue

    soup = BeautifulSoup(response.text, 'html.parser')

    # каждая новость — li внутри ul.triplet
    items = soup.find('ul', class_='triplet').find_all('li')

    for item in items:
        link = item.find('h4').find('a')
        title = link.get_text(strip=True)
        url = DOMAIN + link['href']
        date = item.find('time').get_text(strip=True)

        # ID — число в конце URL: /news/14792/ -> 14792
        news_id = re.search(r'/(\d+)/?$', url).group(1)

        news_list.append({'id': news_id, 'title': title, 'date': date, 'url': url})

    time.sleep(1)
    print(f'Страница {page_num} готова, новостей собрано: {len(news_list)}')

print(f'\nВсего новостей: {len(news_list)}')

In [ ]:
# сохраняем список новостей в news.csv

with open('news.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'title', 'date', 'url'])
    writer.writeheader()
    writer.writerows(news_list)

print(f'Список из {len(news_list)} новостей сохранен в news.csv')

In [ ]:
# для каждой новости парсим детали и сохраняем все в один файл
# просмотры  span.eye
# текст      div.js-mediator-article (или div.content если нет первого)
# теги       a с href содержащим /ru/search/?tag=
# дата      div.news-info-wrapper > time

import os

os.makedirs('news_content', exist_ok=True)

detailed_news = []

for i, news in enumerate(news_list):
    try:
        time.sleep(1)
        response = requests.get(news['url'], timeout=15)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')

        views_tag = soup.find('span', class_='eye')
        views = views_tag.get_text(strip=True) if views_tag else '0'

        # два варианта верстки для текста
        text_tag = soup.find('div', class_='js-mediator-article') or soup.find('div', class_='content')
        text = text_tag.get_text(separator=' ', strip=True) if text_tag else ''

        tag_links = soup.find_all('a', href=lambda h: h and '/ru/search/?tag=' in h)
        tags = ', '.join(t.get_text(strip=True) for t in tag_links)

        date_block = soup.find('div', class_='news-info-wrapper')
        if date_block and date_block.find('time'):
            date = date_block.find('time').get_text(strip=True)
        else:
            date = news['date']

        detailed_news.append({
            'id': news['id'],
            'title': news['title'],
            'date': date,
            'views': views,
            'text': text,
            'tags': tags
        })

    except Exception as e:
        print(f'Ошибка для {news["url"]}: {e}')

    if (i + 1) % 10 == 0:
        print(f'Обработано {i + 1} из {len(news_list)}')

with open('news_content/news_content.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'title', 'date', 'views', 'text', 'tags'])
    writer.writeheader()
    writer.writerows(detailed_news)

print(f'Готово! {len(detailed_news)} новостей сохранены в news_content/news_content.csv')